In [1]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2: Clone Repository
%cd /content
!git clone https://github.com/primakov/DuneAI-Automated-detection-and-segmentation-of-non-small-cell-lung-cancer-computed-tomography-images.git DuneAI
%cd /content/DuneAI

/content
Cloning into 'DuneAI'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 202 (delta 16), reused 14 (delta 14), pack-reused 174 (from 1)
Receiving objects: 100% (202/202), 320.20 MiB | 26.18 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (103/103), done.
/content/DuneAI


In [3]:
# Cell 3: Giải nén dữ liệu từ Drive vào ổ cứng cục bộ của Colab
!mkdir -p /content/DuneAI/dataset
!unzip -q "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/duneai_nrrd_mini.zip" -d "/content/DuneAI/dataset/"

# Kiểm tra xem dữ liệu đã vào đúng thư mục chưa
!ls /content/DuneAI/dataset/duneai_nrrd_mini | head -n 5

LIDC-IDRI-0002
LIDC-IDRI-0005
LIDC-IDRI-0007
LIDC-IDRI-0010
LIDC-IDRI-0018


In [4]:
# Cell 4: Cài đặt Dependencies
!pip install SimpleITK numpy pandas pydicom
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Thử cài đặt pyradiomics từ git repo chính thức hỗ trợ các bản cập nhật mới trước, tránh lỗi build wheel
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git

# Sau đó cài đặt precision-medicine-toolbox
!pip install precision-medicine-toolbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 34.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pydicom]
Looking in indexes: https://download.pytorch.org/whl/cu118
  Cloning https://github.com/AIM-Harvard/pyradiomics.git to /tmp/pip-req-build-8jpcacw3
  Running command git clone --filter=blob:none --quiet https://github.com/AIM-Harvard/pyradiomics.git /tmp/pip-req-build-8jpcacw3
  Resolved https://github.com/AIM-Harvard/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyradiomics: filename=pyradiomics-3.1.1.dev111+g8ed579383-cp313-cp313-lin

In [5]:
# Cell 5: Cài đặt thư viện & Định nghĩa DataLoader
!pip install segmentation-models-pytorch opencv-python

import os
import cv2
import torch
import numpy as np
import SimpleITK as sitk
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

class LungNRRDDataset(Dataset):
    def __init__(self, data_dir, target_size=(256, 256)):
        self.data_dir = data_dir
        self.target_size = target_size
        self.patient_dirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        self.slices_info = []

        print("Đang quét khối 3D để trích xuất các lát cắt chứa khối u...")
        for p_dir in self.patient_dirs:
            mask_path = os.path.join(p_dir, 'mask.nrrd')
            if os.path.exists(mask_path):
                mask_sitk = sitk.ReadImage(mask_path)
                mask = sitk.GetArrayFromImage(mask_sitk)
                # Chỉ lấy những lát cắt dương tính (có điểm ảnh khối u > 0)
                for z in range(mask.shape[0]):
                    if np.sum(mask[z]) > 0:
                        self.slices_info.append((p_dir, z))

        print(f"Hoàn tất! Tổng số lát cắt 2D sẵn sàng để train: {len(self.slices_info)}")

    def __len__(self):
        return len(self.slices_info)

    def __getitem__(self, idx):
        p_dir, z = self.slices_info[idx]

        # Đọc dữ liệu từ file NRRD
        img_vol = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(p_dir, 'image.nrrd')))
        mask_vol = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(p_dir, 'mask.nrrd')))

        # Trích xuất lát cắt 2D theo trục Z
        img_2d = img_vol[z]
        mask_2d = mask_vol[z]

        # Chuẩn hóa kích thước về 256x256 (bắt buộc đối với U-Net)
        img_2d = cv2.resize(img_2d, self.target_size)
        mask_2d = cv2.resize(mask_2d, self.target_size, interpolation=cv2.INTER_NEAREST)

        # Chuyển đổi sang Tensor PyTorch định dạng (C, H, W)
        img_tensor = torch.tensor(img_2d, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask_2d, dtype=torch.float32).unsqueeze(0)

        return img_tensor, mask_tensor

In [6]:
# Cell 6: Khởi tạo Model, Loss, Optimizer
data_dir = "/content/DuneAI/dataset/duneai_nrrd_mini"

# Khởi tạo Dataloader (Batch size 8 phù hợp với VRAM Colab)
train_dataset = LungNRRDDataset(data_dir=data_dir)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

# Khai báo mạng U-Net (dùng ResNet34 làm encoder để hội tụ nhanh)
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,                  # 1 Kênh (Ảnh y tế Grayscale)
    classes=1,                      # 1 Lớp (Khối u)
).to(device)

# Hàm Loss kết hợp Dice Loss tối ưu ranh giới phân đoạn
loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Đang quét khối 3D để trích xuất các lát cắt chứa khối u...
Hoàn tất! Tổng số lát cắt 2D sẵn sàng để train: 2052
Đang sử dụng thiết bị: cuda


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

In [7]:
# Cell 7: Thực thi Training
import time

EPOCHS = 5

print("BẮT ĐẦU HUẤN LUYỆN (DRY-RUN)...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    for i, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)

        # Lan truyền tiến
        optimizer.zero_grad()
        outputs = model(images)

        # Lan truyền ngược
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (i+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f"--> HOÀN TẤT EPOCH {epoch+1} | Trung bình Loss: {epoch_loss:.4f} | Thời gian: {epoch_time:.2f}s\n")

print("Huấn luyện Dry-run thành công! Model Base đã hội tụ.")

BẮT ĐẦU HUẤN LUYỆN (DRY-RUN)...
Epoch [1/5], Batch [10/257], Loss: 0.9983
Epoch [1/5], Batch [20/257], Loss: 0.9978
Epoch [1/5], Batch [30/257], Loss: 0.9969
Epoch [1/5], Batch [40/257], Loss: 0.9983
Epoch [1/5], Batch [50/257], Loss: 0.9964
Epoch [1/5], Batch [60/257], Loss: 0.9993
Epoch [1/5], Batch [70/257], Loss: 0.9972
Epoch [1/5], Batch [80/257], Loss: 0.9985
Epoch [1/5], Batch [90/257], Loss: 0.9982
Epoch [1/5], Batch [100/257], Loss: 0.9955
Epoch [1/5], Batch [110/257], Loss: 0.9961
Epoch [1/5], Batch [120/257], Loss: 0.9953
Epoch [1/5], Batch [130/257], Loss: 0.9926
Epoch [1/5], Batch [140/257], Loss: 0.9974
Epoch [1/5], Batch [150/257], Loss: 0.9932
Epoch [1/5], Batch [160/257], Loss: 0.9977
Epoch [1/5], Batch [170/257], Loss: 0.9952
Epoch [1/5], Batch [180/257], Loss: 0.9870
Epoch [1/5], Batch [190/257], Loss: 0.9974
Epoch [1/5], Batch [200/257], Loss: 0.9956
Epoch [1/5], Batch [210/257], Loss: 0.9900
Epoch [1/5], Batch [220/257], Loss: 0.9886
Epoch [1/5], Batch [230/257], L